# ViT Malware Classifier — Complete Architecture Visualization

This notebook reconstructs the trained `vit_tiny_patch16_224` model (fine-tuned for 2-class malware detection)  
and renders its **complete computational graph** using [`torchview`](https://github.com/mert-kurttutan/torchview).

**Model summary:**
- Backbone: `timm` `vit_tiny_patch16_224` (frozen during training)
- Head: `nn.Linear(192, 2)` — replaced & fine-tuned
- Input: `(1, 3, 224, 224)` — ImageNet-normalised RGB
- Weights: `vit_malware.pth`

**All graphs** are saved to `./architecture_graphs/`

## 0 — Install dependencies (run once)

In [ ]:
# !pip install torchview torchinfo timm --quiet

## 1 — Imports & output directory

In [ ]:
import torch
import torch.nn as nn
import timm
from torchview import draw_graph
import os

# ── All graphs go here ──
GRAPH_DIR = "./architecture_graphs"
os.makedirs(GRAPH_DIR, exist_ok=True)
print(f"Graph output directory : {os.path.abspath(GRAPH_DIR)}")

print(f"PyTorch  : {torch.__version__}")
print(f"timm     : {timm.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {DEVICE}")

## 2 — Reconstruct the model architecture

In [ ]:
model = timm.create_model('vit_tiny_patch16_224', pretrained=False)

num_classes = 2
model.head = nn.Linear(model.head.in_features, num_classes)

print("Model skeleton created.")
print(f"  Embedding dim : {model.embed_dim}")
print(f"  Patch size    : {model.patch_embed.patch_size}")
print(f"  Num patches   : {model.patch_embed.num_patches}")
print(f"  Num blocks    : {len(model.blocks)}")
print(f"  Head out      : {model.head.out_features} classes")

## 3 — Load trained weights

In [ ]:
WEIGHTS_PATH = "vit_malware.pth"
assert os.path.exists(WEIGHTS_PATH), f"Weights file '{WEIGHTS_PATH}' not found."

state_dict = torch.load(WEIGHTS_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()
model = model.to("cpu")

print(f"✅  Loaded '{WEIGHTS_PATH}'")
print(f"   Total parameters   : {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4 — `torchinfo` text summary

In [ ]:
try:
    from torchinfo import summary
    summary(
        model,
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        depth=5,
        device="cpu",
    )
except ImportError:
    print("torchinfo not installed — pip install torchinfo")

## 5 — Full model graph  →  `architecture_graphs/full_model.pdf`

In [ ]:
INPUT_SHAPE = (1, 3, 224, 224)

model_graph = draw_graph(
    model,
    input_size=INPUT_SHAPE,
    device="cpu",
    expand_nested=True,
    show_shapes=True,
    hide_inner_tensors=False,
    hide_module_functions=False,
    graph_name="ViT_Malware_Classifier",
    depth=10,
    save_graph=True,
    filename="full_model",
    directory=GRAPH_DIR,
)
print(f"Saved → {GRAPH_DIR}/full_model.pdf")
model_graph.visual_graph

## 6 — Zoomed sub-graphs  →  `architecture_graphs/`

### 6a — Patch Embedding  →  `patch_embed.pdf`

In [ ]:
g = draw_graph(
    model.patch_embed,
    input_size=(1, 3, 224, 224),
    device="cpu",
    expand_nested=True,
    show_shapes=True,
    graph_name="PatchEmbedding",
    save_graph=True,
    filename="patch_embed",
    directory=GRAPH_DIR,
)
print(f"Saved → {GRAPH_DIR}/patch_embed.pdf")
g.visual_graph

### 6b — Single Transformer Block  →  `transformer_block.pdf`

In [ ]:
NUM_TOKENS = 1 + model.patch_embed.num_patches   # 197
EMBED_DIM  = model.embed_dim                     # 192

g = draw_graph(
    model.blocks[0],
    input_size=(1, NUM_TOKENS, EMBED_DIM),
    device="cpu",
    expand_nested=True,
    show_shapes=True,
    graph_name="TransformerBlock_0",
    save_graph=True,
    filename="transformer_block",
    directory=GRAPH_DIR,
)
print(f"Saved → {GRAPH_DIR}/transformer_block.pdf")
g.visual_graph

### 6c — MLP sub-block (FPGA target)  →  `mlp_block.pdf`

In [ ]:
mlp_block = model.blocks[0].mlp
print(mlp_block)
print(f"  fc1: {mlp_block.fc1.in_features} → {mlp_block.fc1.out_features}")
print(f"  fc2: {mlp_block.fc2.in_features} → {mlp_block.fc2.out_features}")

g = draw_graph(
    mlp_block,
    input_size=(1, NUM_TOKENS, EMBED_DIM),
    device="cpu",
    expand_nested=True,
    show_shapes=True,
    graph_name="MLP_Block_0",
    save_graph=True,
    filename="mlp_block",
    directory=GRAPH_DIR,
)
print(f"Saved → {GRAPH_DIR}/mlp_block.pdf")
g.visual_graph

### 6d — Multi-Head Self-Attention  →  `mhsa_block.pdf`

In [ ]:
g = draw_graph(
    model.blocks[0].attn,
    input_size=(1, NUM_TOKENS, EMBED_DIM),
    device="cpu",
    expand_nested=True,
    show_shapes=True,
    graph_name="MHSA_Block_0",
    save_graph=True,
    filename="mhsa_block",
    directory=GRAPH_DIR,
)
print(f"Saved → {GRAPH_DIR}/mhsa_block.pdf")
g.visual_graph

## 7 — Weight statistics & cross-check vs W1.mem / W2.mem

In [ ]:
import numpy as np

SCALE = 256
W1 = mlp_block.fc1.weight.detach().cpu()
W2 = mlp_block.fc2.weight.detach().cpu()
W1_fixed = (W1 * SCALE).round().int()
W2_fixed = (W2 * SCALE).round().int()

for name, W_f, W_fixed in [("W1 (fc1)", W1, W1_fixed), ("W2 (fc2)", W2, W2_fixed)]:
    print(f"── {name} ────────────")
    print(f"  Shape     : {list(W_f.shape)}")
    print(f"  Float     : [{W_f.min():.4f}, {W_f.max():.4f}]")
    print(f"  Q8.8      : [{W_fixed.min().item()}, {W_fixed.max().item()}]")

for fname, W_fixed in [("W1.mem", W1_fixed), ("W2.mem", W2_fixed)]:
    if os.path.exists(fname):
        mem = np.loadtxt(fname, dtype=int)
        match = np.array_equal(mem, W_fixed.numpy().flatten())
        print(f"\n{fname} cross-check: {'✅ PASS' if match else '❌ MISMATCH'}")
    else:
        print(f"\n{fname}: not found")

## 8 — End-to-end inference sanity check

In [ ]:
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224)
    logits = model(dummy)
    probs  = torch.softmax(logits, dim=-1)

print(f"Input : {list(dummy.shape)}")
print(f"Logits: {logits.squeeze().tolist()}")
print(f"Probs : benign={probs[0,0]:.4f}  malware={probs[0,1]:.4f}")
print(f"Pred  : {'malware' if probs[0,1] > probs[0,0] else 'benign'}")